In [12]:
# This script is taken partly from this study https://github.com/wbsg-uni-mannheim/wdc-lspc-v2
# Similarity features is taken from https://github.com/anhaidgroup/py_entitymatching/blob/master/py_entitymatching/feature/simfunctions.py
# It is simplified for working selected product group.
import numpy as np
import pandas as pd
import argparse
import math
import os
import random
import string
import time
from copy import deepcopy
import nltk
from nltk.corpus import stopwords
from sklearn.ensemble import RandomForestClassifier
from sklearn.base import clone
from sklearn.metrics import classification_report
from sklearn.model_selection import PredefinedSplit, RandomizedSearchCV
import joblib
from pathlib import Path

ROOT = Path.cwd().resolve()
if not (ROOT / "trained_models").is_dir():
    ROOT = next(parent for parent in ROOT.parents if (parent / "trained_models").is_dir())
DATA = ROOT / "data"
OUTPUT = ROOT / "trained_models" / "RF" / "random_forest_v1.joblib"
OUTPUT.parent.mkdir(parents=True, exist_ok=True)
np.random.seed(42)
random.seed(42)

# Number of itertions
N_ITER = 100
#Workers
N_JOBS = 4

# Taken from here https://github.com/wbsg-uni-mannheim/wdc-lspc-v2/blob/master/notebooks/modeling/magellan/01-weyoun2211-run-magellan.ipynb
def classifiers():
    return { 'RandomForest': {'clf': RandomForestClassifier(random_state=42, n_jobs=N_JOBS),
        'params': {'n_estimators': [100],
        'max_features': ['sqrt', 'log2', None],
        'max_depth': [2, 4, 7, 10],
        'min_samples_split': [2, 5, 10, 20],
        'min_samples_leaf': [1, 2, 4, 8],
        'class_weight': [None, 'balanced_subsample']}}}
    
# Clean (we can remove stopwords tokenize and preproces_strings parts
#since we use normalized (clean) data but it improves accuracy %2)
def STOPWORD():
    words = stopwords.words('english')
    return [
        word.lower().translate(str.maketrans('', '', string.punctuation))
        for word in words]

def tokenize(words):
    if isinstance(words, float):
        if words != words:
            return []
    words = str(words)
    words = words.replace('&amp;', '')
    words = words.replace('&reg;', '')
    words = words.replace('&quot;', '')
    words = words.replace('\t', ' ')
    words = words.replace('\n', ' ')
    return words.lower().translate(str.maketrans('', '', string.punctuation)).split()

def preprocess_string(words, stop_words):
    if isinstance(words, float):
        if words != words:
            return words
    word_list = tokenize(words)
    word_list_stopwords_removed = [x for x in word_list if x not in stop_words]
    return ' '.join(word_list_stopwords_removed)

def preprocess_string_column(column, stop_words):
    return column.apply(preprocess_string, args=(stop_words,))

# Similarity score formulas 
# https://github.com/anhaidgroup/py_entitymatching/blob/master/py_entitymatching/feature/simfunctions.py
def _levenshtein(s1, s2):
        if len(s1) < len(s2):
            s1, s2 = s2, s1
        prev = list(range(len(s2) + 1))
        for i, c1 in enumerate(s1, 1):
            cur = [i]
            for j, c2 in enumerate(s2, 1):
                cur.append(min(prev[j] + 1, cur[j - 1] + 1, prev[j - 1] + (c1 != c2)))
            prev = cur
        return float(prev[-1])

def lev_dist(s1, s2):
    if s1 == s2:
        return 0.0
    return _levenshtein(s1, s2)

def lev_sim(s1, s2):
    max_len = max(len(s1), len(s2))
    if max_len == 0:
        return 1.0
    return 1 - (lev_dist(s1, s2) / max_len)

def _jaro_core(s1, s2):
    len1, len2 = len(s1), len(s2)
    search_range = max(len1, len2) // 2 - 1
    if search_range < 0:
        search_range = 0
    flags1 = [False] * len1
    flags2 = [False] * len2
    common = 0
    for i in range(len1):
        low = i - search_range if i > search_range else 0
        high = i + search_range if i + search_range < len2 else len2 - 1
        ch = s1[i]
        for j in range(low, high + 1):
            if not flags2[j] and s2[j] == ch:
                flags1[i] = flags2[j] = True
                common += 1
                break
    if common == 0:
        return 0.0
    trans = 0
    k = 0
    for i in range(len1):
        if flags1[i]:
            j = k
            while j < len2:
                if flags2[j]:
                    k = j + 1
                    break
                j += 1
            if s1[i] != s2[j]:
                trans += 1
    trans = trans // 2
    common = float(common)
    return (common / len1 + common / len2 + (common - trans) / common) / 3.0

def _jaro_winkler_core(s1, s2):
    score = _jaro_core(s1, s2)
    j = min(len(s1), len(s2), 4)
    i = 0
    while i < j and s1[i] == s2[i] and s1[i]:
        i += 1
    if i:
        score += i * 0.1 * (1 - score)
    return score

def jaro_winkler(s1, s2):
    if len(s1) == 0 or len(s2) == 0:
        return 0
    return _jaro_winkler_core(s1, s2)
def needleman_wunsch(s1, s2, gap_cost=1.0):
    n, m = len(s1), len(s2)
    prev = [-(j * gap_cost) for j in range(m + 1)]
    for i in range(1, n + 1):
        cur = [-(i * gap_cost)] + [0.0] * m
        c1 = s1[i - 1]
        for j in range(1, m + 1):
            match = prev[j - 1] + (1.0 if c1 == s2[j - 1] else 0.0)
            cur[j] = max(match, prev[j] - gap_cost, cur[j - 1] - gap_cost)
        prev = cur
    return float(prev[m])
def smith_waterman(s1, s2, gap_cost=1.0):
    n, m = len(s1), len(s2)
    best = 0.0
    prev = [0.0] * (m + 1)
    for i in range(1, n + 1):
        cur = [0.0] * (m + 1)
        c1 = s1[i - 1]
        for j in range(1, m + 1):
            match = prev[j - 1] + (1.0 if c1 == s2[j - 1] else 0.0)
            v = max(0.0, match, prev[j] - gap_cost, cur[j - 1] - gap_cost)
            cur[j] = v
            if v > best:
                best = v
        prev = cur
    return best

def jaccard(arr1, arr2):
    if arr1 == arr2:
        return 1.0
    if len(arr1) == 0 or len(arr2) == 0:
        return 0
    s1, s2 = set(arr1), set(arr2)
    return float(len(s1 & s2)) / float(len(s1 | s2))
def cosine(arr1, arr2):
    if arr1 == arr2:
        return 1.0
    if len(arr1) == 0 or len(arr2) == 0:
        return 0
    s1, s2 = set(arr1), set(arr2)
    return float(len(s1 & s2)) / (math.sqrt(float(len(s1))) * math.sqrt(float(len(s2))))
def monge_elkan(arr1, arr2):
    if arr1 == arr2:
        return 1.0
    if len(arr1) == 0 or len(arr2) == 0:
        return 0
    total = 0.0
    for a in arr1:
        best = float('-inf')
        for b in arr2:
            v = jaro_winkler(a, b)
            if v > best:
                best = v
        total += best
    return total / float(len(arr1))

# Tokening
# https://github.com/anhaidgroup/py_entitymatching/blob/master/py_entitymatching/feature/tokenizers.py
def tok_qgram3(s):
    s = '##' + s + '$$'
    if len(s) < 3:
        return []
    return [s[i:i + 3] for i in range(len(s) - 2)]
def tok_delim_space(s):
    return [t for t in s.split(' ') if t]

TOKENIZERS = {'qgm_3': tok_qgram3, 'dlm_dc0': tok_delim_space}
SIM_FUNCS = {'lev_dist': lev_dist, 'lev_sim': lev_sim, 'jaro_winkler': jaro_winkler,
    'needleman_wunsch': needleman_wunsch, 'smith_waterman': smith_waterman,
    'jaccard': jaccard, 'cosine': cosine, 'monge_elkan': monge_elkan}

FEATURES_PER_TYPE = {
    'str_bt_1w_5w': [  ('jaccard', 'qgm_3', 'qgm_3'), ('cosine', 'dlm_dc0', 'dlm_dc0'),
        ('jaccard', 'dlm_dc0', 'dlm_dc0'), ('monge_elkan',), ('lev_dist',),
        ('lev_sim',), ('needleman_wunsch',), ('smith_waterman',)],
    'str_gt_10w': [ ('jaccard', 'qgm_3', 'qgm_3'), ('cosine', 'dlm_dc0', 'dlm_dc0')]  }

def attr_type(column):
    """Average number of tokens"""
    column = column.dropna()
    avg = column.astype(str).str.split().apply(len).mean()
    if avg <= 5:
        return 'str_bt_1w_5w'
    elif avg <= 10:
        return 'str_bt_5w_10w'
    return 'str_gt_10w'
def get_features(left_table, right_table, attributes):
    """Returns a list of (feature (name) , (attr)ibute, (sim) name, (tok)enizer name)."""
    features = []
    for attr in attributes:
        lt, rt = attr_type(left_table[attr]), attr_type(right_table[attr])
        if lt != rt:
            # Magellan skips an attribute whose left/right types differ
            continue
        for f in FEATURES_PER_TYPE[lt]:
            sim = f[0] 
            tok = (f[1] if len(f) == 3 else None)
            name = '_'.join([attr, attr, sim] + ([tok, tok] if tok else []))
            features.append((name, attr, sim, tok))
    return features
def extract_feature_vecs(pairs, features):
    """pairs has attr_left attr_right"""
    out = {}
    cache = {}
    for name, attr, sim, tok in features:
        left = pairs[attr + '_left'].tolist()
        right = pairs[attr + '_right'].tolist()
        fn = SIM_FUNCS[sim]
        if tok:
            key = (attr, tok)
            if key not in cache:
                t = TOKENIZERS[tok]
                cache[key] = ([t(x) for x in left], [t(x) for x in right])
            left, right = cache[key]
        elif sim in ('monge_elkan',):
            left = [[x] for x in left]
            right = [[x] for x in right]
        out[name] = [fn(a, b) for a, b in zip(left, right)]
    return pd.DataFrame(out, index=pairs.index)
# Load data
preprocess_features = ['title', 'description', 'brand', 'specTableContent']
def load_pairs(path, stop_words):
    df = pd.read_json(path, compression='gzip', lines=True)
    df = df.where(df.notna(), '')
    for col in preprocess_features:
        for side in ('_left', '_right'):
            if col + side not in df.columns:
                df[col + side] = np.nan
            df[col + side] = preprocess_string_column(df[col + side], stop_words)
    for col in preprocess_features:
        for side in ('_left', '_right'):
            c = df[col + side]
            c = c.where(~c.isin(['', 'nan', 'null']), np.nan)
            df[col + side] = c.fillna('').astype(str)
    return df
def side_table(df, side):
    cols = [c for c in df.columns if c.endswith(side)]
    t = df[cols].drop_duplicates(subset='id' + side).copy()
    t.columns = [c[:-len(side)] for c in cols]
    return t

# Train
# https://github.com/wbsg-uni-mannheim/wdc-lspc-v2/blob/master/notebooks/modeling/magellan/01-weyoun2211-run-magellan.ipynb
FEATURE_COMBINATIONS = [['title', 'description', 'brand', 'specTableContent']]
def run_magellan(train_df, valid_ids, gs_df, feature_combinations, classifiers, n_iter=N_ITER):

    results = []
    A_t = side_table(train, '_left')
    B_t = side_table(train, '_right')

    for feature_combination in feature_combinations:
        feature_report = '+'.join(feature_combination)
        F_t = get_features(A_t, B_t, feature_combination)

        H_t = extract_feature_vecs(train_df, F_t)
        H_t['label'] = train_df['label'].values
        H_t['pair_id'] = train_df['pair_id'].values
        H_gs = extract_feature_vecs(gs_df, F_t)
        H_gs['label'] = gs_df['label'].values

        val_df = H_t[H_t['pair_id'].isin(valid_ids)].drop(columns='pair_id')
        train_only_df = H_t[~H_t['pair_id'].isin(valid_ids)].drop(columns='pair_id')
        train_only_df = train_only_df.sample(frac=1, random_state=42)
        train_all = pd.concat([train_only_df, val_df])

        # -1 = train  0 = validation
        test_fold = np.r_[np.full(len(train_only_df), -1), np.zeros(len(val_df), dtype=int)]
        ps = PredefinedSplit(test_fold=test_fold)
        X = train_all.drop(columns='label')
        y = train_all['label']
        X_gs = H_gs.drop(columns='label')
        y_gs = H_gs['label']

        for model_name, model_spec in classifiers.items():
            search = RandomizedSearchCV(
                estimator=model_spec['clf'],
                param_distributions=model_spec['params'],
                random_state=42,
                n_jobs=N_JOBS,
                scoring='f1',
                n_iter=n_iter,
                cv=ps,
                pre_dispatch=8,
                return_train_score=True )
            search.fit(X, y)
            learner = clone(search.best_estimator_)
            X_train = train_only_df.drop(columns='label')
            y_train = train_only_df['label']
            start = time.time()
            learner.fit(X_train, y_train)

            joblib.dump({'model': learner, 'features': F_t, 'feature_columns': X_train.columns.tolist() }, OUTPUT)
            
            train_time = time.time() - start
            start = time.time()
            predictions = learner.predict(X_gs)
            prediction_time = time.time() - start
            metrics = classification_report(y_gs, predictions, output_dict=True, zero_division=0)
            positive = metrics.get('1', {'precision': 0, 'recall': 0, 'f1-score': 0})
            f1 = positive['f1-score']
            best_index = search.best_index_
            results.append({
                'feature': feature_report,
                'model': model_name,
                'mean_train_score': search.cv_results_['mean_train_score'][best_index],
                'mean_valid_score': search.cv_results_['mean_test_score'][best_index],
                'f1_test': f1,
                'precision_test': positive['precision'],
                'recall_test': positive['recall'],
                'best_params': search.best_params_,
                'train_time': train_time,
                'prediction_time': prediction_time     })
    return pd.DataFrame(results)

In [13]:
train = load_pairs(DATA / 'computers_train_xlarge.json.gz', STOPWORD())
gs    = load_pairs(DATA / 'computers_gs.json.gz', STOPWORD())
valid_ids = pd.read_csv(DATA / 'computers_valid_xlarge.csv')['pair_id'].values
print(len(train), 'training |', len(valid_ids), 'validation |', 'gold standard', len(gs))

68461 training | 13693 validation | gold standard 1100


In [14]:
#Feature table
A = side_table(train, '_left')
B = side_table(train, '_right')
F = get_features(A, B, preprocess_features)
X_train = extract_feature_vecs(train, F)
X_train['label'] = train['label'].values
X_train.head()

,title_title_jaccard_qgm_3_qgm_3,title_title_cosine_dlm_dc0_dlm_dc0,description_description_jaccard_qgm_3_qgm_3,description_description_cosine_dlm_dc0_dlm_dc0,brand_brand_jaccard_qgm_3_qgm_3,brand_brand_cosine_dlm_dc0_dlm_dc0,brand_brand_jaccard_dlm_dc0_dlm_dc0,brand_brand_monge_elkan,brand_brand_lev_dist,brand_brand_lev_sim,brand_brand_needleman_wunsch,brand_brand_smith_waterman,specTableContent_specTableContent_jaccard_qgm_3_qgm_3,specTableContent_specTableContent_cosine_dlm_dc0_dlm_dc0,label
0,0.080000,0.272166,0.000000,0.000000,0.0,0.0,0.0,0.0,13.0,0.0,-13.0,0.0,0.117207,0.129619,1
1,0.536232,0.866025,0.981481,1.000000,1.0,1.0,1.0,1.0,0.0,1.0,13.0,13.0,1.000000,1.000000,1
2,0.359375,0.603023,0.000000,0.000000,1.0,1.0,1.0,1.0,0.0,1.0,-0.0,0.0,1.000000,1.000000,1
3,0.283951,0.492366,0.000000,0.000000,0.0,0.0,0.0,0.0,4.0,0.0,-4.0,0.0,0.000000,0.000000,1
4,0.354839,0.707107,0.108280,0.235702,0.0,0.0,0.0,0.0,4.0,0.0,-4.0,0.0,1.000000,1.000000,1


In [15]:
## train, results
results = run_magellan(train, valid_ids, gs, FEATURE_COMBINATIONS, classifiers(), n_iter=N_ITER)
display(results[['model', 'mean_valid_score', 'f1_test']])

,model,mean_valid_score,f1_test
0,RandomForest,0.599099,0.644567


In [18]:
# Load random_forest_v1.joblib
data = joblib.load(OUTPUT)

model = data['model']
features = data['features']
feature_names = data['feature_columns']